# Fine-tuning BAT on data subsets 

### Import libraries and load in config file

In [1]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var

from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *
from icu_benchmarks.models.dl_models.bat import * 

from icu_benchmarks.models.train import load_model
from pathlib import Path

import torch
import random
import numpy as np

vars_dict = {
    "GROUP": "stay_id",
    "SEQUENCE": "time",
    "LABEL": "label",
    "DYNAMIC": ["alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun", "ca", "cai", "ck", "ckmb", "cl",
        "crea", "crp", "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact", "lymph", "map", "mch", "mchc", "mcv",
        "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos", "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine",
        "wbc"],
    "STATIC": ["age", "sex", "height", "weight"],
}

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin")

/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

#### Load in pre-trained model

In [2]:
# Pre-trained on pooled mimic + miiv
#model_path = Path("/work3/s185395/yaib_logs/mimic_miiv/LOS/SSL_BAT_tuned_mimic_miiv/2025-11-17T11-33-11/repetition_0/fold_0/model.ckpt")
# Pre-trained on pooled eicu + mimic
#model_path = Path("/work3/s185395/yaib_logs/eicu_mimic/LOS/SSL_BAT_tuned_eicu_mimic/2025-11-18T01-30-40/repetition_0/fold_0/model.ckpt")
# Pre-trained on pooled eicu + miiv
model_path = Path("/work3/s185395/yaib_logs/eicu_miiv/LOS/SSL_BAT_tuned_eicu_miiv/2025-11-17T18-49-46/repetition_0/fold_0/model.ckpt")

ckpt = torch.load(model_path, map_location="cpu")
hparams = ckpt.get("hyper_parameters", {})

# Instantiate the model class (init args can be anything required)
model = SSL_BAT(**hparams) 

# Load only encoder weights
encoder_state_dict = {k.replace("model.encoder_class.", ""): v
                      for k, v in ckpt["state_dict"].items()
                      if k.startswith("model.encoder_class.")}

model.model.encoder_class.load_state_dict(encoder_state_dict)

# Extract encoder from SSL_BAT
pretrained_encoder = model.model.encoder_class

# Create classification model using the pretrained encoder
classification_model = EncoderPrediction(
    encoder_class=pretrained_encoder,
    prediction_head=BinaryClassificationHead,
    prediction_head_kwargs={"num_classes": 2}
)

use static


/tmp/ipykernel_3635655/1913325240.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location="cpu")


#### Load in saved out preprocessed data subsets

In [3]:
import polars as pl
from pathlib import Path

dataset = 'mimic' # eicu, miiv, mimic
size = 100 # 100, 500, 1000, 2000, 3000, 5000, 7000, 9000. 9506
seed = 42 # 42, 84, 126, 168, 210 
subset_path = f"/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/{dataset}/{size}_{seed}" # !Change dataset subset here! 

# Set the directory where your Parquet files are saved
dir = Path(subset_path)

# Create the data dictionary in the format returned by preprocess_data()
data = {}

for split in ["train", "val", "test"]:
    outcome_path = dir / f"{split}_OUTCOME.parquet"
    features_path = dir / f"{split}_FEATURES.parquet"

    if outcome_path.exists() and features_path.exists():
        data[split] = {
            "OUTCOME": pl.read_parquet(outcome_path),
            "FEATURES": pl.read_parquet(features_path),
        }
        print(f"✅ Loaded {split} data")
    else:
        print(f"⚠️ Missing files for split '{split}'")


✅ Loaded train data
✅ Loaded val data
✅ Loaded test data


#### Create train, val and test datasets

In [4]:
from copy import deepcopy
from tqdm import tqdm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

from torch.utils.data import random_split
from icu_benchmarks.data.loader import *
from torch.utils.data import DataLoader

finetune_train_set = BATPolarsDataset(data=data, split="train", ram_cache=False, runmode=RunMode.classification, vars=vars_dict)
finetune_val_set = BATPolarsDataset(data=data, split="val", ram_cache=False, runmode=RunMode.classification,vars=vars_dict)
finetune_test_set = BATPolarsDataset(data=data, split="test", ram_cache=False, runmode=RunMode.classification, vars=vars_dict)

#### Function for fine-tuning on data subsets

In [5]:
def run_experiment(bz, lr, model_path, fine_tune_head, num_epochs = 200):

    # Setting seed for reproducibility (Only want variability in the subset datasets)
    seed = 42

    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # If using CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    g = torch.Generator()
    g.manual_seed(seed)

    bz = bz
    lr = lr

    ckpt = torch.load(model_path, map_location="cpu")
    hparams = ckpt.get("hyper_parameters", {})

    # Reset model parameters 
    # Instantiate the model class (init args can be anything required)
    model = SSL_BAT(**hparams) 

    # Load only encoder weights
    encoder_state_dict = {k.replace("model.encoder_class.", ""): v
                        for k, v in ckpt["state_dict"].items()
                        if k.startswith("model.encoder_class.")}

    model.model.encoder_class.load_state_dict(encoder_state_dict)

    # Extract encoder from SSL_BAT
    pretrained_encoder = model.model.encoder_class

    # Create classification model using the pretrained encoder
    classification_model = EncoderPrediction(
        encoder_class=pretrained_encoder,
        prediction_head=BinaryClassificationHead,
        prediction_head_kwargs={"num_classes": 2}
    )

    finetune_train_loader = DataLoader(finetune_train_set, batch_size=bz, shuffle=True, generator=g, 
                                    collate_fn=finetune_train_set.collate_fn_pad_to_longest_in_batch())
    finetune_val_loader = DataLoader(finetune_val_set, batch_size=bz, shuffle=True, generator=g, 
                                    collate_fn=finetune_val_set.collate_fn_pad_to_longest_in_batch())
    eval_loader = DataLoader(finetune_test_set, batch_size=bz, shuffle=False,
                            collate_fn=finetune_test_set.collate_fn_pad_to_longest_in_batch())

    print(f'Finetuening training dataset length: {len(finetune_train_set)}')
    print(f'Finetuening validation dataset length: {len(finetune_val_set)}')
    print(f'Finetuening test dataset length: {len(finetune_test_set)}')

    # Automatically select device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🔧 Using device: {device}")

    if fine_tune_head:
        for param in classification_model.parameters():
            param.requires_grad = False
        for param in classification_model.head.parameters():
            param.requires_grad = True
    else:
        for param in classification_model.parameters():
            param.requires_grad = True

    classification_model.to(device)
    optimizer = torch.optim.Adam(classification_model.parameters(), lr=lr)

    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

    loss_fn = torch.nn.CrossEntropyLoss()

    train_losses, val_losses = [], []
    train_aurocs, val_aurocs = [], []
    train_auprcs, val_auprcs = [], []

    # Set early stopping parameters
    patience = 3
    best_val_auprc = 0
    epochs_without_improvement = 0
    best_model_state = None

    for epoch in range(num_epochs):
        #current_lr = optimizer.param_groups[0]['lr']
        #print(f"📉 Current LR after epoch {epoch+1}: {current_lr:.6f}")
        # ======== TRAINING ========
        classification_model.train()
        total_train_loss = 0
        all_train_labels = []
        all_train_probs = []

        loop = tqdm(finetune_train_loader, desc=f"🔧 Fine-tuning Epoch {epoch+1}/{num_epochs}")
        for batch in loop:
            x, mask, label, times, static, *_ = batch
            x = x.to(device).float()
            mask = mask.to(device).float()
            times = times.to(device).float()
            static = static.to(device).float()
            label = label.to(device).long()

            optimizer.zero_grad()
            logits = classification_model(x, static=static, time=times, sensor_mask=mask)
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            probs = F.softmax(logits, dim=1)[:, 1]  # Probabilities for class 1

            all_train_labels.extend(label.cpu().numpy())
            all_train_probs.extend(probs.detach().cpu().numpy())
            loop.set_postfix(loss=loss.item())

        avg_train_loss = total_train_loss / len(finetune_train_loader)
        train_losses.append(avg_train_loss)

        train_auroc = roc_auc_score(all_train_labels, all_train_probs)
        train_auprc = average_precision_score(all_train_labels, all_train_probs)
        train_aurocs.append(train_auroc)
        train_auprcs.append(train_auprc)

        #print(f"✅ Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f} | AUROC: {train_auroc:.4f} | AUPRC: {train_auprc:.4f}")

        # ======== VALIDATION ========
        classification_model.eval()
        total_val_loss = 0
        all_val_labels = []
        all_val_probs = []

        with torch.no_grad():
            for batch in finetune_val_loader:
                x, mask, label, times, static, *_ = batch
                x = x.to(device).float()
                mask = mask.to(device).float()
                times = times.to(device).float()
                static = static.to(device).float()
                label = label.to(device).long()

                logits = classification_model(x, static=static, time=times, sensor_mask=mask)
                loss = loss_fn(logits, label)
                total_val_loss += loss.item()

                probs = F.softmax(logits, dim=1)[:, 1]
                all_val_labels.extend(label.cpu().numpy())
                all_val_probs.extend(probs.cpu().numpy())

        avg_val_loss = total_val_loss / len(finetune_val_loader)
        val_losses.append(avg_val_loss)

        val_auroc = roc_auc_score(all_val_labels, all_val_probs)
        val_auprc = average_precision_score(all_val_labels, all_val_probs)
        val_aurocs.append(val_auroc)
        val_auprcs.append(val_auprc)

        print(f"🧪 Validation — Loss: {avg_val_loss:.4f} | AUROC: {val_auroc:.4f} | AUPRC: {val_auprc:.4f}")

        # ======== EARLY STOPPING & BEST MODEL SAVE ========
        scheduler.step()  # update learning rate based on val AUPRC

        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_model_state = deepcopy(classification_model.state_dict())
            epochs_without_improvement = 0
            #print(f"📌 New best AUPRC: {best_val_auprc:.4f} — model checkpoint saved")
        else:
            epochs_without_improvement += 1
            #print(f"⏳ No AUPRC improvement for {epochs_without_improvement} epoch(s)")

        if epochs_without_improvement >= patience:
            #print(f"🛑 Early stopping triggered after {patience} epochs without improvement.")
            break

    # Restore best model after training
    classification_model.load_state_dict(best_model_state)

    # ======== TESTING ========
    #print("\n🚀 Starting evaluation on test set...")

    classification_model.eval()
    total_test_loss = 0
    all_test_labels = []
    all_test_probs = []

    with torch.no_grad():
        test_loop = tqdm(eval_loader, desc="🧪 Evaluating on Test Set")
        for batch in test_loop:
            x, mask, label, times, static, *_ = batch
            x = x.to(device).float()
            mask = mask.to(device).float()
            times = times.to(device).float()
            static = static.to(device).float()
            label = label.to(device).long()

            logits = classification_model(x, static=static, time=times, sensor_mask=mask)
            loss = loss_fn(logits, label)
            total_test_loss += loss.item()

            probs = F.softmax(logits, dim=1)[:, 1]
            all_test_labels.extend(label.cpu().numpy())
            all_test_probs.extend(probs.cpu().numpy())

            test_loop.set_postfix(loss=loss.item())

    avg_test_loss = total_test_loss / len(eval_loader)
    test_auroc = roc_auc_score(all_test_labels, all_test_probs)
    test_auprc = average_precision_score(all_test_labels, all_test_probs)

    print(f"\n🎯 Test Set Results:")
    print(f"   Loss : {avg_test_loss:.4f}")
    print(f"   AUROC: {test_auroc:.4f}")
    print(f"   AUPRC: {test_auprc:.4f}")

    return {'bz': bz, 'lr': lr,'avg_test_loss': avg_test_loss, 'test_auroc': test_auroc, 'test_auprc': test_auprc}

### Grid hyperparameter tuning

In [6]:
# Full model or only head tuning 
#fine_tune_head = True
fine_tune_head = False

In [ ]:

# Learning rates and batch sizes to test
lrs = [0.0005]
batch_sizes = [64]

# Store results
results = []

# Run the experiment for each (bz, lr) pair
for bz in batch_sizes:
    for lr in lrs:
        print(f"\n🚀 Running experiment with batch_size={bz}, learning_rate={lr}")
        result = run_experiment(bz, lr, model_path, fine_tune_head, num_epochs = 100) 
        results.append(result)



🚀 Running experiment with batch_size=8, learning_rate=0.0005
use static
Finetuening training dataset length: 100
Finetuening validation dataset length: 2377
Finetuening test dataset length: 2971
🔧 Using device: cuda


/tmp/ipykernel_3635655/3769831599.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location="cpu")


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 15.77 GiB of which 256.00 KiB is free. Process 3584834 has 13.99 GiB memory in use. Including non-PyTorch memory, this process has 1.77 GiB memory in use. Of the allocated memory 1.33 GiB is allocated by PyTorch, and 75.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Test specific combinations of batch size and lr 

##### MIMIC head 

In [25]:
results

[{'bz': 64,
  'lr': 0.0045,
  'avg_test_loss': 0.29687057942786116,
  'test_auroc': 0.7979258328095538,
  'test_auprc': 0.3869372102431376},
 {'bz': 64,
  'lr': 0.0055,
  'avg_test_loss': 0.3004661427533373,
  'test_auroc': 0.8004017979656695,
  'test_auprc': 0.39205953089606504}]

In [22]:
results

[{'bz': 64,
  'lr': 0.003,
  'avg_test_loss': 0.30042916250989793,
  'test_auroc': 0.7932140296208425,
  'test_auprc': 0.38063492137068744},
 {'bz': 64,
  'lr': 0.006,
  'avg_test_loss': 0.3063112005908438,
  'test_auroc': 0.8011902337975052,
  'test_auprc': 0.39439462137539394}]

In [20]:
results

[{'bz': 64,
  'lr': 0.002,
  'avg_test_loss': 0.2984457418639609,
  'test_auroc': 0.7982190756101663,
  'test_auprc': 0.3893820832677789}]

In [18]:
results

[{'bz': 64,
  'lr': 0.004,
  'avg_test_loss': 0.2904882405666595,
  'test_auroc': 0.8074612034241907,
  'test_auprc': 0.4082012827435516}]

In [10]:
results

[{'bz': 64,
  'lr': 0.05,
  'avg_test_loss': 0.2487767329860118,
  'test_auroc': 0.8393927195074045,
  'test_auprc': 0.3689607827746173}]

In [8]:
results

[{'bz': 64,
  'lr': 0.007,
  'avg_test_loss': 0.21691799822353547,
  'test_auroc': 0.8440429849376028,
  'test_auprc': 0.3844479061103139},
 {'bz': 64,
  'lr': 0.01,
  'avg_test_loss': 0.23386575570029597,
  'test_auroc': 0.8460416807077816,
  'test_auprc': 0.3860330898849343}]

##### MIMIC full

In [ ]:
LR 5e-05: AUROC mean=0.7456, sd=0.0000, AUPRC mean=0.3041, sd=0.0000

LR 9e-05: AUROC mean=0.7725, sd=0.0000, AUPRC mean=0.3209, sd=0.0000

LR 0.0005: AUROC mean=0.8163, sd=0.0000, AUPRC mean=0.3497, sd=0.0000

LR 0.001: AUROC mean=0.8303, sd=0.0000, AUPRC mean=0.3647, sd=0.0000

LR 0.005: AUROC mean=0.8410, sd=0.0000, AUPRC mean=0.3815, sd=0.0000

In [ ]:
LR 5e-05: AUROC mean=0.7456, sd=0.0000, AUPRC mean=0.3041, sd=0.0000

LR 9e-05: AUROC mean=0.7725, sd=0.0000, AUPRC mean=0.3209, sd=0.0000

LR 0.0005: AUROC mean=0.8163, sd=0.0000, AUPRC mean=0.3497, sd=0.0000

LR 0.001: AUROC mean=0.8303, sd=0.0000, AUPRC mean=0.3647, sd=0.0000

LR 0.005: AUROC mean=0.8410, sd=0.0000, AUPRC mean=0.3815, sd=0.0000

In [31]:
results

[{'bz': 64,
  'lr': 0.0002,
  'avg_test_loss': 0.28313663760398294,
  'test_auroc': 0.8182115951523092,
  'test_auprc': 0.4194936383296956},
 {'bz': 64,
  'lr': 8.5e-05,
  'avg_test_loss': 0.28670266112114523,
  'test_auroc': 0.8228581103212612,
  'test_auprc': 0.4208313850456455}]

In [29]:
results

[{'bz': 64,
  'lr': 8e-05,
  'avg_test_loss': 0.28589078435238374,
  'test_auroc': 0.8250911819123414,
  'test_auprc': 0.4249660620170617},
 {'bz': 64,
  'lr': 0.00095,
  'avg_test_loss': 0.36070068529311644,
  'test_auroc': 0.5673982613468366,
  'test_auprc': 0.15345770698293884}]

In [27]:
results

[{'bz': 64,
  'lr': 6e-05,
  'avg_test_loss': 0.28417498602512037,
  'test_auroc': 0.8250236807393702,
  'test_auprc': 0.41114717733812906},
 {'bz': 64,
  'lr': 9e-05,
  'avg_test_loss': 0.2813338975005962,
  'test_auroc': 0.8268274006072892,
  'test_auprc': 0.429459899163755}]

In [25]:
results

[{'bz': 64,
  'lr': 9e-06,
  'avg_test_loss': 0.2856797401575332,
  'test_auroc': 0.8164045555545719,
  'test_auprc': 0.40509668600046145},
 {'bz': 64,
  'lr': 7e-05,
  'avg_test_loss': 0.28835006470375873,
  'test_auroc': 0.8189004390895971,
  'test_auprc': 0.4201365911446035}]

In [23]:
results

[{'bz': 64,
  'lr': 1e-05,
  'avg_test_loss': 0.2859082710235677,
  'test_auroc': 0.8164078752843902,
  'test_auprc': 0.40751495228231965},
 {'bz': 64,
  'lr': 5e-05,
  'avg_test_loss': 0.28579980737351357,
  'test_auroc': 0.8235961969175201,
  'test_auprc': 0.4141113416416715}]

In [21]:
results

[{'bz': 64,
  'lr': 0.0001,
  'avg_test_loss': 0.2868938182896756,
  'test_auroc': 0.8213703180743797,
  'test_auprc': 0.41933520283893133},
 {'bz': 64,
  'lr': 0.005,
  'avg_test_loss': 0.36130238720711244,
  'test_auroc': 0.5652216251626667,
  'test_auprc': 0.1514084320098315},
 {'bz': 64,
  'lr': 0.001,
  'avg_test_loss': 0.36063280194363695,
  'test_auroc': 0.5675974451359319,
  'test_auprc': 0.1526367466061278},
 {'bz': 64,
  'lr': 0.05,
  'avg_test_loss': 1.689182163553035,
  'test_auroc': 0.558594337868822,
  'test_auprc': 0.1339680585780296},
 {'bz': 64,
  'lr': 0.01,
  'avg_test_loss': 0.3581556303704039,
  'test_auroc': 0.5756300847195049,
  'test_auprc': 0.14521324970714142}]

In [ ]:
top_configs = [
    #{'bz': 64, 'lr': 6e-3},
    #{'bz': 24, 'lr': 1.5e-2},
    {'bz': 64, 'lr': 1e-3},
]

results = []

for config in top_configs:
    print(f"\n🚀 Re-running experiment: bz={config['bz']}, lr={config['lr']}")
    result = run_experiment(config['bz'], config['lr'], model_path, fine_tune_head)
    results.append(result)


### Optimal hyperparameters when hyperparameter tuend for each subset size


In [ ]:
import polars as pl
from pathlib import Path
from copy import deepcopy
from tqdm import tqdm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.utils.data import random_split
from icu_benchmarks.data.loader import *
from torch.utils.data import DataLoader


hp_results = []
seeds = [42, 84, 126, 168, 210] 
for seed in seeds: 

    dataset = 'mimic' # eicu, miiv, mimic
    size = 500 # 100, 500, 1000, 2000, 3000, 5000, 7000, 9000. 9506
    subset_path = f"/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/{dataset}/{size}_{seed}" # !Change dataset subset here! 

    # Set the directory where your Parquet files are saved
    dir = Path(subset_path)

    # Create the data dictionary in the format returned by preprocess_data()
    data = {}

    for split in ["train", "val", "test"]:
        outcome_path = dir / f"{split}_OUTCOME.parquet"
        features_path = dir / f"{split}_FEATURES.parquet"

        if outcome_path.exists() and features_path.exists():
            data[split] = {
                "OUTCOME": pl.read_parquet(outcome_path),
                "FEATURES": pl.read_parquet(features_path),
            }
            print(f"✅ Loaded {split} data")
        else:
            print(f"⚠️ Missing files for split '{split}'")

    finetune_train_set = BATPolarsDataset(data=data, split="train", ram_cache=False, runmode=RunMode.classification, vars=vars_dict)
    finetune_val_set = BATPolarsDataset(data=data, split="val", ram_cache=False, runmode=RunMode.classification,vars=vars_dict)
    finetune_test_set = BATPolarsDataset(data=data, split="test", ram_cache=False, runmode=RunMode.classification, vars=vars_dict)

    # Full model or only head tuning 
    #fine_tune_head = True
    fine_tune_head = False

    # Learning rates and batch sizes to test
    lrs = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]
    batch_sizes = [64]

    # Store results
    results = []

    # Run the experiment for each (bz, lr) pair
    for bz in batch_sizes:
        for lr in lrs:
            print(f"\n🚀 Running experiment with batch_size={bz}, learning_rate={lr}")
            result = run_experiment(bz, lr, model_path, fine_tune_head, num_epochs = 200) 
            results.append(result)

    hp_results.append(results)
